# Breakdown Harness Engineering in Claude Code

## Initialization and Configuration

In [ ]:
import os
import subprocess

from pathlib import Path
from anthropic import Anthropic
from dotenv import load_dotenv

ImportError: cannot import name 'path' from 'pathlib' (/Users/rong/.local/share/mise/installs/python/3.12.13/lib/python3.12/pathlib.py)

In [ ]:
"""
Get Environment Variable
1. ANTHROPIC_API_KEY
2. ANTHROPIC_BASE_URL
3. ANTHROPIC_MODEL_ID
"""

# override覆蓋.env中的同名變數
load_dotenv(override=True)

WORKDIR = Path.cwd()
# getenv 不一定要有； environ 是一定要有否則無法執行
client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url=os.getenv("ANTHROPIC_BASE_URL"),
)
MODEL = os.environ["ANTHROPIC_MODEL_ID"]
MAX_TOKEN = 8000

# getcwd = get current working directory (= pwd) -> 讓LLM知道目前工作目錄
SYSTEM = f"You are a coding agent at {os.getcwd()}. Use bash to solve tasks. Act, don't explain."

TOOLS = [{
    "name": "bash",
    "description": "Run a shell command",
    "input_schema": {
        "type": "object",
        "properties": {
            "command": 
                {"type": "string"}
            },
        "required": ["command"],
    }
}]

## S02 Tools

> "加一个工具, 只加一个 handler" -- 循环不用动, 新工具注册进 dispatch map 就行。

只有 bash 时, 所有操作都走 shell。cat 截断不可预测, sed 遇到特殊字符就崩, 每次 bash 调用都是不受约束的安全面。专用工具 (read_file, write_file) 可以在工具层面做路径沙箱。

关键洞察: 加工具不需要改循环

**Process**

```text
+--------+      +-------+      +------------------+
|  User  | ---> |  LLM  | ---> | Tool Dispatch    |
| prompt |      |       |      | {                |
+--------+      +---+---+      |   bash: run_bash |
                    ^           |   read: run_read |
                    |           |   write: run_wr  |
                    +-----------+   edit: run_edit |
                    tool_result | }                |
                                +------------------+

The dispatch map is a dict: {tool_name: handler_function}.
One lookup replaces any if/elif chai

In [ ]:
def safe_path(p: str) -> Path:
    """
    Get path of working directory safely
    """
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

In [ ]:
def run_bash(command: str) -> str:
    """
    Tool-1: run bash
    """
    # Risky commands
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command will be blocked"
    try:
        # 讓LLM執行外部指令
        # 標準輸出存進 r.stdout，錯誤輸出存進 r.stderr
        r = subprocess.run(command,
                           shell=True,
                           cwd=os.getcwd(),
                           capture_output=True,
                           text=True,
                           timeout=120
                           )
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout(120s)"

In [ ]:
def run_read(path: str, limit: int = None) -> str:
    """
    Tool 2: Read the file
    """
    try: 
        text = safe_path(path).read_text()
        lines = text.splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def run_write(path: str, content: str) -> str:
    """
    Tool 3: Write the file
    """
    try:
        fp = safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content)
        return f"Work {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def run_edit(path: str, old_text: str, new_text: str) -> str:
    """
    Tool 4: Edit the file
    """
    try:
        fp = safe_path(path)
        content = fp.read_text()
        if old_text not in content:
            return f"Error: Text not found in {path}"
        fp.write.text(content.replace(old_text, new_text, 1))
        return f"Edited {path}"
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# -- The dispatch map: {tool_name: handler} --
TOOL_HANDLERS = {
    "bash": lambda **kw: run_bash(kw["command"]),
    "read_file": lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file": lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"])
}

TOOLS = [
    {"name": "bash", "description": "Run a shell command.",
     "input_schema": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}},
    {"name": "read_file", "description": "Read file contents.",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["path"]}},
    {"name": "write_file", "description": "Write content to file.",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]}},
    {"name": "edit_file", "description": "Replace exact text in file.",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}, "old_text": {"type": "string"}, "new_text": {"type": "string"}}, "required": ["path", "old_text", "new_text"]}},
]

## S01 Agent Loop

> 一个退出条件控制整个流程。循环持续运行, 直到模型不再调用工具。

**Process**

```text
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> |  Tool   |
| prompt |      |       |      | execute |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                    (loop until stop_reason != "tool_use")
```

1. 先用 load_dotenv(override=True) 載入 .env
2. 用 os.environ / os.getenv() 取得 API 設定
3. 用 os.getcwd() 知道目前專案目錄
4. 使用者在終端機輸入問題 `message: {"role": "user"}`
5. 模型如果要用工具，就要求執行 bash `def run_bash`
6. subprocess.run(...) 在目前資料夾執行指令
7. 把指令結果再送回模型
8. 模型輸出最終文字答案 `message: {"role": "assistant"}`
9. 只把 text block 印出來 `print(block.text)`

In [ ]:
def agent_loop(messages):
    """
    The main agent loop with bash
    """
    while True:
        response = client.messages.create(
            model=MODEL,
            system=SYSTEM,
            messages=messages,
            tools=TOOLS,
            max_tokens=MAX_TOKEN
        )
        # role = assistant 主角是「模型」
        messages.append({
            "role": "assistant",
            "content": response.content
        })

        # 模型要你先去執行工具，再把結果送回來，如果沒有要執行工具，就結束對話
        if response.stop_reason != "tool_use":
            return

        results = []
        for block in response.content:
            if block.type == "tool_use":
                handler = TOOL_HANDLERS.get(block.name)
                output = handler(**block.input) if handler else f"Unknown tool: {block.name}"
                print(f"> {block.name}: {output[:200]}")
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": output,
                })

        # role = user 主角是「用戶」
        messages.append({
            "role": "user",
            "content": results
        })

In [ ]:
if __name__ == "__main__":
    """
    Main Process
    """
    history = []
    while True:
        try:
            # 印出彩色提示字，然後等待你輸入內容
            query = input("\033[36ms01 >> \033[0m")
        # 提前終止程序 (Ctrl + C 或是 Ctrl + D終止)
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() in ("q", "exit", ""):
            break
        history.append({
            "role": "user",
            "content": query
        })
        agent_loop(history)
        response_content = history[-1]["content"]
        if isinstance(response_content, list):
            # 逐一檢查模型回傳的 content blocks，把其中的純文字內容印出來。
            for block in response_content:
                if hasattr(block, "text"):
                    print(block.text)
        print()

Hello! I'm ready to help you with coding tasks. I'm currently in the `/Users/rong/Workspaces/2-Areas/21-Claude-Code/210-Harness-Agents` directory and can execute bash commands to assist you.

What would you like me to do?

There is 1 Python file in this directory:
- `main.py`

